# 05 Build the model tables

Builds the tables that the semantic model reads, in this Lakehouse. Run it after notebooks 01 and 02, and
again whenever the model should show newer live data; **Run**, **Schedule** runs it by itself.

- **One continuous 10-minute series.** `fact_scada_10min` holds the generated history and, on top of it, the
  live telemetry of the Eventhouse rolled up to 10-minute periods with the same columns and the same
  definitions. Where a complete live period exists, it replaces the generated one.
- **Dimension tables** for dates, turbines and the site.
- **Fact tables** for events, work orders, curtailments, the budget, market prices, the forecast and the met mast.

Attach the Lakehouse `WindFarmLH` as the default Lakehouse before running.

In [ ]:
import json
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")                 # every time in these tables is UTC

with open("/lakehouse/default/Files/reference/site.json", encoding="utf-8") as f:
    SITE = json.load(f)
MODEL = SITE["turbineModel"]
CUT_IN, RATED_WIND, CUT_OUT = MODEL["cutInWindSpeed"], MODEL["ratedWindSpeed"], MODEL["cutOutWindSpeed"]
RATED_KW = MODEL["ratedPowerKw"]
STANDARD_AIR_DENSITY = 1.225

SIGNALS = ["WindSpeed", "AmbientTemp", "ActivePower", "PowerSetpoint", "GearboxOilTemp", "GearboxBearingTemp",
           "GearboxVibration", "GeneratorWindingTemp", "MainBearingTemp"]


def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)
    print(f"{name:24s} {spark.table(name).count():>9,} rows")


def has_rows(name):
    return spark.catalog.tableExists(name) and spark.table(name).limit(1).count() == 1


def text(query):                 # one value as text; timestamps stay in UTC that way
    return spark.sql(query).first()[0]


AS_OF = text("SELECT date_format(max(Timestamp), 'yyyy-MM-dd HH:mm:ss') FROM ops_scada_10min")
print("History ends at", AS_OF, "UTC")

## Live telemetry, rolled up to 10-minute periods

The same rules as the history: a period's state is the state it was in for most of the time; possible power
comes from the power curve at the air density of the moment; lost energy is possible minus delivered energy
when the turbine was stopped or limited. Only complete periods are used. The period that is still filling is
left for the next run. Periods without telemetry, for example while the VM was stopped, become rows with the
category `Information unavailable`.

In [ ]:
LIVE = has_rows("TurbineTelemetry")
if not LIVE:
    print("No live telemetry in the Lakehouse (shortcut TurbineTelemetry missing or empty): history only.")
else:
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW live_bins AS
        SELECT TurbineId, timestamp_seconds(floor(unix_timestamp(Timestamp) / 600) * 600) AS Timestamp,
               count(*) AS Samples, mode(State) AS State, """
              + ", ".join(f"avg({c}) AS {c}" for c in SIGNALS) + """
        FROM TurbineTelemetry
        GROUP BY TurbineId, timestamp_seconds(floor(unix_timestamp(Timestamp) / 600) * 600)""")
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW live_density AS
        SELECT timestamp_seconds(floor(unix_timestamp(Timestamp) / 600) * 600) AS Timestamp,
               avg(AirDensity) AS AirDensity, avg(WindSpeed) AS WindSpeed, avg(AmbientTemp) AS AmbientTemp
        FROM WeatherObservations
        GROUP BY timestamp_seconds(floor(unix_timestamp(Timestamp) / 600) * 600)""" if has_rows("WeatherObservations") else """
        CREATE OR REPLACE TEMP VIEW live_density AS
        SELECT CAST(NULL AS timestamp) AS Timestamp, CAST(NULL AS double) AS AirDensity,
               CAST(NULL AS double) AS WindSpeed, CAST(NULL AS double) AS AmbientTemp WHERE 1 = 0""")
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW live_10min AS
        WITH lim AS (SELECT max(Timestamp) AS Filling, max(Samples) AS FullSamples FROM live_bins),
        site AS (SELECT coalesce(avg(AirDensity), {STANDARD_AIR_DENSITY}) AS MeanDensity FROM live_density),
        b AS (
            SELECT l.*, coalesce(w.AirDensity, site.MeanDensity) AS AirDensity,
                   least(1.0, l.Samples / lim.FullSamples) AS Coverage
            FROM live_bins l CROSS JOIN lim CROSS JOIN site
            LEFT JOIN live_density w ON w.Timestamp = l.Timestamp
            WHERE l.Timestamp < lim.Filling),
        p AS (
            SELECT b.*, WindSpeed * pow(AirDensity / {STANDARD_AIR_DENSITY}, 1.0 / 3.0) AS EqWind FROM b),
        q AS (
            SELECT p.*,
                   CASE WHEN EqWind < {CUT_IN} OR EqWind > {CUT_OUT} THEN 0.0
                        ELSE {RATED_KW} * (pow(least(greatest(EqWind, {CUT_IN}), {RATED_WIND}), 3) - pow({CUT_IN}, 3))
                             / (pow({RATED_WIND}, 3) - pow({CUT_IN}, 3)) END AS PossiblePower,
                   CASE WHEN Coverage < 0.5 THEN 'Information unavailable'
                        WHEN State = 'RUNNING' THEN 'Full performance'
                        WHEN State = 'DERATED' THEN 'Partial performance'
                        WHEN State = 'CURTAILED' AND ActivePower <= 0.01 * {RATED_KW} THEN 'Requested shutdown'
                        WHEN State = 'CURTAILED' THEN 'Partial performance'
                        WHEN State IN ('IDLE', 'STOPPED_HIGH_WIND') THEN 'Out of environmental specification'
                        WHEN State = 'PAUSED' THEN 'Technical standby'
                        WHEN State = 'STOPPED' THEN 'Requested shutdown'
                        WHEN State = 'MAINTENANCE' THEN 'Scheduled maintenance'
                        WHEN State = 'FAULT' THEN 'Forced outage'
                        ELSE 'Information unavailable' END AS AvailabilityCategory,
                   CASE WHEN Coverage < 0.5 OR State = 'RUNNING' THEN NULL
                        WHEN State = 'DERATED' THEN 'Technical derate'
                        WHEN State = 'CURTAILED' THEN 'Grid curtailment'
                        WHEN State = 'IDLE' THEN 'Low wind'
                        WHEN State = 'STOPPED_HIGH_WIND' THEN 'High wind'
                        WHEN State = 'PAUSED' THEN 'Cable untwist'
                        WHEN State = 'STOPPED' THEN 'Operator stop'
                        WHEN State = 'MAINTENANCE' THEN 'Scheduled maintenance'
                        WHEN State = 'FAULT' THEN 'Component fault' END AS LossCause
            FROM p)
        SELECT Timestamp, CAST(Timestamp AS date) AS Date, TurbineId, 'Live' AS Source, State, AvailabilityCategory,
               LossCause, """ + ", ".join(f"round({c}, 2) AS {c}" for c in SIGNALS[:2]) + """, round(AirDensity, 4) AS AirDensity,
               """ + ", ".join(f"round({c}, 2) AS {c}" for c in SIGNALS[2:4]) + """, round(PossiblePower, 1) AS PossiblePower,
               """ + ", ".join(f"round({c}, 2) AS {c}" for c in SIGNALS[4:]) + """,
               round(greatest(ActivePower, 0) / 6 * Coverage, 3) AS EnergyKwh,
               round(PossiblePower / 6 * Coverage, 3) AS PossibleEnergyKwh,
               round(CASE WHEN LossCause IS NULL OR LossCause = 'Low wind' THEN 0.0
                          ELSE greatest(PossiblePower - greatest(ActivePower, 0), 0) / 6 * Coverage END, 3) AS LostEnergyKwh
        FROM q""")
    LIVE_END = text("SELECT date_format(max(Timestamp), 'yyyy-MM-dd HH:mm:ss') FROM live_10min")
    LIVE = LIVE_END is not None
    print("Live periods:", text("SELECT count(DISTINCT Timestamp) FROM live_10min"), "complete 10-minute periods from",
          text("SELECT date_format(min(Timestamp), 'yyyy-MM-dd HH:mm:ss') FROM live_10min"), "to", LIVE_END, "UTC")

## The continuous series

History first; live periods replace generated ones where both exist; periods after the end of the history in
which a turbine sent nothing are added as `Information unavailable`.

In [ ]:
COLUMNS = ["Timestamp", "Date", "TurbineId", "Source", "State", "AvailabilityCategory", "LossCause", "WindSpeed",
           "AmbientTemp", "AirDensity", "ActivePower", "PowerSetpoint", "PossiblePower", "GearboxOilTemp",
           "GearboxBearingTemp", "GearboxVibration", "GeneratorWindingTemp", "MainBearingTemp", "EnergyKwh",
           "PossibleEnergyKwh", "LostEnergyKwh"]
history = spark.table("ops_scada_10min").withColumn("Date", F.col("Timestamp").cast("date")).withColumn("Source", F.lit("History"))
if LIVE:
    live = spark.table("live_10min").cache()
    history = history.join(live.select("TurbineId", "Timestamp"), ["TurbineId", "Timestamp"], "left_anti")
    scada = history.select(*COLUMNS).unionByName(live.select(*COLUMNS))
    if LIVE_END > AS_OF:             # periods after the history in which a turbine sent nothing
        gaps = spark.sql(f"""
            SELECT g.Timestamp, CAST(g.Timestamp AS date) AS Date, t.turbineId AS TurbineId, 'Live' AS Source,
                   'UNKNOWN' AS State, 'Information unavailable' AS AvailabilityCategory
            FROM (SELECT explode(sequence(TIMESTAMP'{AS_OF}' + INTERVAL 10 MINUTES, TIMESTAMP'{LIVE_END}',
                                          INTERVAL 10 MINUTES)) AS Timestamp) g
            CROSS JOIN ref_turbines t
            LEFT ANTI JOIN live_10min l ON l.TurbineId = t.turbineId AND l.Timestamp = g.Timestamp""")
        scada = scada.unionByName(gaps, allowMissingColumns=True).select(*COLUMNS)
else:
    scada = history.select(*COLUMNS)
save(scada, "fact_scada_10min")

met_columns = ["Timestamp", "Date", "Source", "WindSpeed", "AmbientTemp", "AirDensity"]
met = (spark.table("ops_met_10min").withColumn("Date", F.col("Timestamp").cast("date"))
       .withColumn("Source", F.lit("History")).select(*met_columns))
if LIVE:
    met = met.unionByName(spark.sql(f"""
        SELECT Timestamp, CAST(Timestamp AS date) AS Date, 'Live' AS Source, round(WindSpeed, 2) AS WindSpeed,
               round(AmbientTemp, 2) AS AmbientTemp, round(AirDensity, 4) AS AirDensity
        FROM live_density
        WHERE Timestamp > TIMESTAMP'{AS_OF}' AND Timestamp < (SELECT max(Timestamp) FROM live_density)"""))
save(met, "fact_met_10min")

## Dimension tables

In [ ]:
FIRST_DAY = text("""SELECT CAST(CAST(date_trunc('year', least((SELECT min(Timestamp) FROM ops_scada_10min),
                                                              (SELECT min(OpenedDate) FROM ops_work_orders))) AS date) AS string)""")
LAST_DAY = text("SELECT CAST(make_date(year(max(Date)) + 1, 12, 31) AS string) FROM fact_scada_10min")
save(spark.sql(f"""
    SELECT d AS Date, year(d) AS Year, concat('Q', quarter(d)) AS Quarter, month(d) AS MonthNo,
           date_format(d, 'MMM') AS Month, date_format(d, 'yyyy-MM') AS YearMonth,
           CAST(date_trunc('month', d) AS date) AS MonthStart,
           CASE WHEN month(d) >= 7 THEN concat(year(d), '/', lpad(CAST((year(d) + 1) % 100 AS string), 2, '0'))
                ELSE concat(year(d) - 1, '/', lpad(CAST(year(d) % 100 AS string), 2, '0')) END AS ContractYear,
           (dayofweek(d) + 5) % 7 + 1 AS DayOfWeekNo, date_format(d, 'EEE') AS DayOfWeek
    FROM (SELECT explode(sequence(DATE'{FIRST_DAY}', DATE'{LAST_DAY}', INTERVAL 1 DAY)) AS d)"""), "dim_date")

save(spark.sql("""
    SELECT turbineId AS TurbineId, name AS Name, model AS Model, CAST(ratedPowerKw AS double) AS RatedPowerKw,
           `string` AS String, CAST(latitude AS double) AS Latitude, CAST(longitude AS double) AS Longitude,
           CAST(commissionedDate AS date) AS CommissionedDate
    FROM ref_turbines"""), "dim_turbine")

save(spark.sql("""
    SELECT siteId AS SiteId, siteName AS SiteName, timezone AS Timezone, CAST(capacityMw AS double) AS CapacityMw,
           CAST(ppaPriceUsdPerMwh AS double) AS PpaPriceUsdPerMwh,
           CAST(gridEmissionFactorKgCo2PerMwh AS double) AS GridEmissionFactorKgCo2PerMwh
    FROM ref_site"""), "dim_site")

save(spark.sql("SELECT 1 AS KpiId"), "kpi")                          # the table that carries the measures

## Fact tables

Each one has a date column for the relationship to the date table. Preventive work orders get a result
(`PmResult`) as question MP-02 defines it; market prices and the day-ahead forecast are joined with what the
site produced in the same hour.

In [ ]:
save(spark.sql("""
    SELECT EventId, TurbineId, StartTime, EndTime, CAST(StartTime AS date) AS StartDate, DurationHours,
           AvailabilityCategory, Cause, Component, AlarmCode, FailureMode, ResponsibleParty, Description, LostEnergyMwh
    FROM ops_turbine_events"""), "fact_turbine_events")

save(spark.sql(f"""
    SELECT WorkOrderId, TurbineId, WorkType, Priority, Status, CAST(OpenedDate AS date) AS OpenedDate,
           CAST(ScheduledDate AS date) AS ScheduledDate, CAST(ClosedDate AS date) AS ClosedDate, Summary,
           LaborHours, TotalCostUsd, DowntimeHours, WaitingForPartsHours, CostBorneBy,
           CASE WHEN WorkType <> 'Preventive' OR ScheduledDate IS NULL OR ScheduledDate >= TIMESTAMP'{AS_OF}' THEN NULL
                WHEN Status = 'Closed' AND ClosedDate <= ScheduledDate + INTERVAL 7 DAYS THEN 'Done within a week of the plan'
                WHEN Status = 'Closed' THEN 'Done late'
                ELSE 'Not done, overdue' END AS PmResult
    FROM ops_work_orders"""), "fact_work_orders")

save(spark.sql("""
    SELECT CurtailmentId, StartTime, CAST(StartTime AS date) AS StartDate, DurationHours, Reason, InstructedBy,
           CASE WHEN Compensated THEN 'Paid' ELSE 'Not paid' END AS Compensation, LostEnergyMwh, AvgRealTimePriceUsdPerMwh
    FROM ops_curtailments"""), "fact_curtailments")

save(spark.sql("""
    SELECT YearMonth, to_date(concat(YearMonth, '-01')) AS MonthStart, P50EnergyMwh, P90EnergyMwh, LongTermWindSpeed,
           BudgetRevenueUsd, BudgetCorrectiveMaintenanceUsd, BudgetOpexUsd
    FROM ops_monthly_budget"""), "fact_monthly_budget")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW site_hourly AS
    SELECT date_trunc('hour', Timestamp) AS Hour, sum(EnergyKwh) / 1000 AS SiteEnergyMwh,
           sum(PossiblePower) / 6 / 1000 AS PossiblePowerMw, sum(ActivePower) / 6 / 1000 AS ActualPowerMw
    FROM fact_scada_10min GROUP BY date_trunc('hour', Timestamp)""")

save(spark.sql("""
    SELECT p.Timestamp, CAST(p.Timestamp AS date) AS Date, p.DayAheadPriceUsdPerMwh, p.RealTimePriceUsdPerMwh,
           h.SiteEnergyMwh
    FROM ops_market_prices p JOIN site_hourly h ON h.Hour = p.Timestamp"""), "fact_market_hourly")

save(spark.sql("""
    SELECT f.TargetTime, CAST(f.TargetTime AS date) AS Date, f.PowerMw AS ForecastPowerMw, f.PowerMwP10, f.PowerMwP90,
           h.PossiblePowerMw, h.ActualPowerMw, abs(f.PowerMw - h.PossiblePowerMw) AS AbsErrorMw
    FROM ops_power_forecast f JOIN site_hourly h ON h.Hour = f.TargetTime
    WHERE f.ForecastType = 'Day-ahead'"""), "fact_forecast_hourly")

## Check

The series must have one row per turbine and period, with no period missing between the first and the last.
The second result shows where the history hands over to the live data.

In [ ]:
check = spark.sql("""
    SELECT count(*) AS Rows, count(DISTINCT TurbineId, Timestamp) AS DistinctRows, count(DISTINCT TurbineId) AS Turbines,
           count(DISTINCT Timestamp) AS Periods, date_format(min(Timestamp), 'yyyy-MM-dd HH:mm') AS FirstPeriod,
           date_format(max(Timestamp), 'yyyy-MM-dd HH:mm') AS LastPeriod,
           CAST((unix_timestamp(max(Timestamp)) - unix_timestamp(min(Timestamp))) / 600 + 1 AS bigint) AS PeriodsExpected
    FROM fact_scada_10min""").first()
print(check.asDict(), "(times in UTC)")
assert check.Rows == check.DistinctRows, "a turbine has two rows for the same period"
assert check.Periods == check.PeriodsExpected, "periods are missing from the series"
assert check.Rows == check.Periods * check.Turbines, "a turbine is missing in some periods"
display(spark.sql("""
    SELECT Source, AvailabilityCategory, count(*) AS Periods, date_format(min(Timestamp), 'yyyy-MM-dd HH:mm') AS FirstPeriod,
           date_format(max(Timestamp), 'yyyy-MM-dd HH:mm') AS LastPeriod,
           round(sum(EnergyKwh) / 1000, 1) AS EnergyMwh
    FROM fact_scada_10min WHERE Date >= date_sub(current_date(), 3) OR Source = 'Live'
    GROUP BY Source, AvailabilityCategory ORDER BY Source, Periods DESC"""))